# C3-14 · QLoRA fine-tuning of Mistral 7B Instruct v0.3 — Lead Intelligence Agent
**Student:** Shathursini J (IT23164062) · **Component:** C3 · **Platform:** Google Colab (free T4 GPU)

**Before you start:** *Runtime → Change runtime type → T4 GPU → Save.* Then *Runtime → Run all*.
Put `train.jsonl` and `val.jsonl` (from `python finetune_data.py`) in Google Drive → `MyDrive/c3_finetune/`
(or upload them when cell 2 asks).

**What is fine-tuned:** behaviour, not facts — valid JSON only, cite the retrieved S-labels including the
follow-up policy, follow the tenant's policy for the lead's segment. Tenant facts still come from RAG.

**Safe on a limited GPU:** everything is saved to Google Drive as it runs (base evaluation, checkpoints every
20 steps, adapter). If Colab disconnects, reconnect and *Run all* again — finished parts are skipped and
training resumes from the last checkpoint.

| Cell | What | Time on T4 |
| --- | --- | --- |
| 1 | install libraries | ~2 min |
| 2 | GPU check, Drive, data | ~1 min |
| 3 | download + load Mistral 7B in 4-bit | ~5 min |
| 4 | tokenise data, mask the prompt | <1 min |
| 5–6 | evaluation helpers + **before** evaluation | ~8 min |
| 7 | QLoRA training (2 epochs) | ~25–40 min |
| 8 | **after** evaluation + comparison | ~8 min |
| 9 | save adapter + downloads | ~1 min |

In [ ]:
# 1. Install (the versions Colab already has for torch are kept)
%pip install -q -U "transformers>=4.46" "peft>=0.13" "accelerate>=1.0" "bitsandbytes>=0.45" "datasets>=3.0"

In [ ]:
# 2. Setup: GPU, Google Drive, data, settings
import os, sys, json, re, time, glob, shutil, inspect
import torch
import pandas as pd

TEST = os.environ.get("C3_TEST") == "1"          # offline dry-run only; leave as is
IN_COLAB = "google.colab" in sys.modules
if not TEST:
    assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU > Save, then Run all again."
    print("GPU:", torch.cuda.get_device_name(0))

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/c3_finetune"
else:
    WORK = os.environ.get("C3_WORK", "c3_finetune")
os.makedirs(WORK, exist_ok=True)

def _have_data():
    return all(os.path.exists(f"{WORK}/{n}") for n in ("train.jsonl", "val.jsonl"))

if not _have_data() and IN_COLAB:
    from google.colab import files
    print("Select BOTH train.jsonl and val.jsonl")
    for name, data in files.upload().items():
        target = "train.jsonl" if "train" in name else "val.jsonl" if "val" in name else None
        if target:
            open(f"{WORK}/{target}", "wb").write(data)
assert _have_data(), f"train.jsonl and val.jsonl must be in {WORK}"

MODEL_ID = os.environ.get("C3_MODEL", "unsloth/mistral-7b-instruct-v0.3")   # open copy of mistralai/Mistral-7B-Instruct-v0.3
MAX_LEN = 1536        # tokens per example (prompts are ~900)
MAX_NEW = 220         # tokens generated per answer (same as rag.py)
N_EVAL = 3 if TEST else 30
EPOCHS, LR, LORA_R, SEED = 2, 2e-4, 16, 42
torch.manual_seed(SEED)
print("Work folder:", WORK)
print({n: sum(1 for _ in open(f"{WORK}/{n}", encoding="utf-8")) for n in ("train.jsonl", "val.jsonl")})

In [ ]:
# 3. Load the tokenizer and Mistral 7B in 4-bit (QLoRA base)
import transformers
from packaging import version
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
print("transformers", transformers.__version__)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if not tokenizer.chat_template:                      # official Mistral v0.3 template, in case the repo has none
    tokenizer.chat_template = ("{{ bos_token }}{% for m in messages %}{% if m['role'] == 'user' %}"
                               "{{ '[INST] ' + m['content'] + '[/INST]' }}{% elif m['role'] == 'assistant' %}"
                               "{{ ' ' + m['content'] + eos_token }}{% endif %}{% endfor %}")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.unk_token or tokenizer.eos_token
tokenizer.padding_side = "right"

# non-quantised parts (embeddings, norms, lm_head) in float32: the T4 has no bfloat16
dtype_kw = {"dtype": torch.float32} if version.parse(transformers.__version__) >= version.parse("4.56") \
    else {"torch_dtype": torch.float32}
if TEST:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **dtype_kw)
else:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map={"": 0}, **dtype_kw)
model.config.use_cache = True
print(f"model loaded: {model.get_memory_footprint() / 1e9:.1f} GB")

In [ ]:
# 4. Tokenise: the loss is computed on the answer only (prompt tokens are masked with -100)
from datasets import load_dataset
raw = load_dataset("json", data_files={"train": f"{WORK}/train.jsonl", "validation": f"{WORK}/val.jsonl"})

def encode(ex):
    msgs = ex["messages"]
    prompt = tokenizer.apply_chat_template(msgs[:1], tokenize=False, add_generation_prompt=True)
    full = tokenizer.apply_chat_template(msgs, tokenize=False)
    p_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    f_ids = tokenizer(full, add_special_tokens=False)["input_ids"]
    return {"input_ids": f_ids, "attention_mask": [1] * len(f_ids),
            "labels": [-100] * len(p_ids) + f_ids[len(p_ids):],
            "prefix_ok": f_ids[:len(p_ids)] == p_ids, "n_tokens": len(f_ids)}

tok = raw.map(encode, remove_columns=["messages"])
for split in ("train", "validation"):
    t = tok[split]
    assert all(t["prefix_ok"]), "prompt/answer split failed - send this message to Claude"
    print(split, len(t), "examples | tokens median", int(pd.Series(t["n_tokens"]).median()), "max", max(t["n_tokens"]))
tok = tok.filter(lambda e: e["n_tokens"] <= MAX_LEN).remove_columns(["prefix_ok", "n_tokens"])
print("after length filter:", {k: len(v) for k, v in tok.items()})

In [ ]:
# 5. Evaluation: the same checks as rag.py (ground + policy_check), on the validation prompts
STOP = {"the","a","an","get","gets","go","into","with","and","to","of","in","on","same","day","lead","leads"}

def check(prompt, text):
    labels = set(re.findall(r"^(S\d+):", prompt, re.M))
    rule = re.search(r"leads (.+?) \(source (S\d+)\)", prompt)
    try:
        out = json.loads(text[text.index("{"): text.rindex("}") + 1])
        assert isinstance(out, dict)
    except Exception:
        return {"json_valid": False, "grounded": False, "policy_consistent": False, "cites_policy": False}
    raw_src = out.get("sources", [])
    raw_src = raw_src if isinstance(raw_src, list) else [raw_src]
    srcs = [str(s).strip("[] ").upper() for s in raw_src]
    srcs = [s for s in srcs if s in labels]
    action = str(out.get("next_best_action") or "")
    words = {w for w in re.findall(r"[a-z]+", rule.group(1).lower()) if w not in STOP and len(w) > 3} if rule else set()
    return {"json_valid": True, "grounded": bool(srcs) and bool(action),
            "policy_consistent": any(w in action.lower() for w in words),
            "cites_policy": bool(rule) and rule.group(2) in srcs}

@torch.no_grad()
def generate(prompt):
    text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
    enc = tokenizer(text, add_special_tokens=False, return_tensors="pt").to(model.device)
    out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)

def evaluate(tag):
    path = f"{WORK}/c3-14_eval_{tag}.csv"
    if os.path.exists(path):                       # already done before a disconnect
        print("loaded saved", path)
        return pd.read_csv(path)
    model.eval()
    model.config.use_cache = True
    val = raw["validation"].select(range(min(N_EVAL, len(raw["validation"]))))
    rows = []
    for i, ex in enumerate(val):
        prompt = ex["messages"][0]["content"]
        t0 = time.time()
        text = generate(prompt)
        rows.append({"i": i, **check(prompt, text), "seconds": round(time.time() - t0, 2), "model": tag,
                     "output": text[:600]})
        print(f"{tag}: {i + 1}/{len(val)}", end="\r")
    df = pd.DataFrame(rows)
    df.to_csv(path, index=False)
    print("\n", tag, df.drop(columns=["i"]).mean(numeric_only=True).round(3).to_dict())
    return df

In [ ]:
# 6. BEFORE fine-tuning: base Mistral 7B on the validation prompts
before = evaluate("base")
print(before.iloc[0]["output"])

In [ ]:
# 7. QLoRA training (resumes from the last checkpoint after a disconnect)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
from transformers.trainer_utils import get_last_checkpoint

ADAPTER_DIR = f"{WORK}/c3-mistral-lora"
if os.path.exists(f"{ADAPTER_DIR}/adapter_config.json"):
    from peft import PeftModel
    print("adapter already trained - loading it from Drive")
    model = PeftModel.from_pretrained(model, ADAPTER_DIR)
    trainer = None
else:
    if not TEST:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                                gradient_checkpointing_kwargs={"use_reentrant": False})
    model.config.use_cache = False
    model = get_peft_model(model, LoraConfig(
        r=LORA_R, lora_alpha=2 * LORA_R, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
    model.print_trainable_parameters()

    steps = -(-len(tok["train"]) // 8) * EPOCHS              # optimiser steps (batch 1 x accumulation 8)
    kw = dict(output_dir=f"{WORK}/checkpoints", num_train_epochs=EPOCHS, learning_rate=LR,
              per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=8,
              lr_scheduler_type="cosine", warmup_steps=max(1, steps // 20), logging_steps=5,
              eval_steps=20, save_strategy="steps", save_steps=20, save_total_limit=2,
              fp16=not TEST, optim="adamw_torch" if TEST else "paged_adamw_8bit",
              report_to="none", seed=SEED)
    if TEST:
        kw.update(max_steps=4, eval_steps=2, save_steps=2, logging_steps=1)
    sig = inspect.signature(TrainingArguments.__init__).parameters
    kw["eval_strategy" if "eval_strategy" in sig else "evaluation_strategy"] = "steps"
    dropped = [k for k in kw if k not in sig]                # settings this transformers version does not know
    if dropped:
        print("skipping settings not supported by this transformers version:", dropped)
    args = TrainingArguments(**{k: v for k, v in kw.items() if k in sig})
    print(f"training: {len(tok['train'])} examples, {steps} optimiser steps")
    trainer = Trainer(model=model, args=args, train_dataset=tok["train"], eval_dataset=tok["validation"],
                      data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100))
    last = get_last_checkpoint(args.output_dir) if os.path.isdir(args.output_dir) else None
    print("resuming from", last) if last else print("starting training")
    trainer.train(resume_from_checkpoint=last)
    model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    pd.DataFrame(trainer.state.log_history).to_csv(f"{WORK}/c3-14_training_log.csv", index=False)
    print("adapter saved to", ADAPTER_DIR)

In [ ]:
# 8. AFTER fine-tuning: same validation prompts, then the comparison table
after = evaluate("fine-tuned")
cols = ["json_valid", "grounded", "policy_consistent", "cites_policy", "seconds"]
table = pd.concat([before, after]).groupby("model")[cols].mean().round(3).reindex(["base", "fine-tuned"])
print(table)
table.to_csv(f"{WORK}/c3-14_finetune_eval.csv")
summary = {"base_model": MODEL_ID, "method": "QLoRA 4-bit nf4, LoRA r=%d alpha=%d, %d epochs, lr %g" % (LORA_R, 2 * LORA_R, EPOCHS, LR),
           "train_examples": len(tok["train"]), "eval_prompts": len(after),
           "before": table.loc["base"].to_dict(), "after": table.loc["fine-tuned"].to_dict()}
log_path = f"{WORK}/c3-14_training_log.csv"
if os.path.exists(log_path):
    log = pd.read_csv(log_path)
    if "loss" in log:
        summary["train_loss_first_last"] = [round(float(log["loss"].dropna().iloc[0]), 4), round(float(log["loss"].dropna().iloc[-1]), 4)]
    if "eval_loss" in log:
        summary["eval_loss_first_last"] = [round(float(log["eval_loss"].dropna().iloc[0]), 4), round(float(log["eval_loss"].dropna().iloc[-1]), 4)]
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        ax = log.dropna(subset=["loss"]).plot(x="step", y="loss", label="train loss")
        if "eval_loss" in log:
            log.dropna(subset=["eval_loss"]).plot(x="step", y="eval_loss", label="validation loss", ax=ax)
        ax.set_title("C3-14 QLoRA training loss"); ax.set_xlabel("step")
        plt.savefig(f"{WORK}/c3-14_loss.png", dpi=120, bbox_inches="tight")
    except Exception as e:
        print("loss plot skipped:", e)
json.dump(summary, open(f"{WORK}/c3-14_finetune_summary.json", "w"), indent=2)
print(json.dumps(summary, indent=2))

In [ ]:
# 9. Package the adapter and download the results (everything is also in Drive: MyDrive/c3_finetune)
shutil.make_archive(f"{WORK}/c3-mistral-lora", "zip", f"{WORK}/c3-mistral-lora")
for f in sorted(glob.glob(f"{WORK}/c3-14_*")) + [f"{WORK}/c3-mistral-lora.zip"]:
    print(f"{os.path.getsize(f) / 1e6:8.1f} MB  {f}")
if IN_COLAB:
    from google.colab import files
    for f in ["c3-14_finetune_eval.csv", "c3-14_finetune_summary.json", "c3-14_training_log.csv",
              "c3-14_eval_base.csv", "c3-14_eval_fine-tuned.csv", "c3-14_loss.png", "c3-mistral-lora.zip"]:
        if os.path.exists(f"{WORK}/{f}"):
            files.download(f"{WORK}/{f}")
print("Next: File > Download > Download .ipynb  (save as notebooks/02_finetune_mistral_qlora.ipynb)")